# TRABALHO 3 — PARTE 1: Algoritmo Naïve Bayes

Aplicação do algoritmo Gaussian Naïve Bayes à base de risco de crédito.

## Importação das bibliotecas

In [1]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import GaussianNB

## Leitura e avaliação inicial da base

In [2]:
# O CSV deve estar na mesma pasta do notebook.
caminho_csv = Path('risco_credito.csv')
if not caminho_csv.exists() and Path('Parte-1/risco_credito.csv').exists():
    caminho_csv = Path('Parte-1/risco_credito.csv')
if not caminho_csv.exists():
    raise FileNotFoundError('Coloque o arquivo risco_credito.csv na mesma pasta do notebook.')

dataset_risco_credito = pd.read_csv(caminho_csv)
dataset_risco_credito

,historia,divida,garantias,renda,risco
0,ruim,alta,nenhuma,0_15,alto
1,desconhecida,alta,nenhuma,15_35,alto
2,desconhecida,baixa,nenhuma,15_35,moderado
3,desconhecida,baixa,nenhuma,acima_35,alto
4,desconhecida,baixa,nenhuma,acima_35,baixo
5,desconhecida,baixa,adequada,acima_35,baixo
6,ruim,baixa,nenhuma,0_15,alto
7,ruim,baixa,adequada,acima_35,moderado
8,boa,baixa,nenhuma,acima_35,baixo
9,boa,alta,adequada,acima_35,baixo


In [3]:
print(f'Dimensões da base: {dataset_risco_credito.shape[0]} linhas e {dataset_risco_credito.shape[1]} colunas')
print('Valores ausentes por coluna:')
display(dataset_risco_credito.isnull().sum().to_frame('quantidade'))
print('Distribuição da classe risco:')
display(dataset_risco_credito['risco'].value_counts().to_frame('quantidade'))

Dimensões da base: 14 linhas e 5 colunas
Valores ausentes por coluna:


,quantidade
historia,0
divida,0
garantias,0
renda,0
risco,0


Distribuição da classe risco:


,quantidade
risco,
alto,6
baixo,5
moderado,3


# 1 — Pré-processamento dos dados

## a) Divisão da base

As quatro primeiras colunas formam a matriz de atributos `X_risco_credito`; a última coluna é a classe `y_risco_credito`.

In [4]:
X_risco_credito = dataset_risco_credito.iloc[:, 0:4].values
y_risco_credito = dataset_risco_credito.iloc[:, 4].values

print('Formato de X:', X_risco_credito.shape)
print('Formato de y:', y_risco_credito.shape)

Formato de X: (14, 4)
Formato de y: (14,)


## b) Aplicação do Label Encoder

Cada atributo categórico recebe seu próprio codificador. A variável-alvo permanece com os nomes das classes para facilitar a leitura das previsões.

In [5]:
label_encoder_historia = LabelEncoder()
label_encoder_divida = LabelEncoder()
label_encoder_garantias = LabelEncoder()
label_encoder_renda = LabelEncoder()

X_risco_credito[:, 0] = label_encoder_historia.fit_transform(X_risco_credito[:, 0])
X_risco_credito[:, 1] = label_encoder_divida.fit_transform(X_risco_credito[:, 1])
X_risco_credito[:, 2] = label_encoder_garantias.fit_transform(X_risco_credito[:, 2])
X_risco_credito[:, 3] = label_encoder_renda.fit_transform(X_risco_credito[:, 3])
X_risco_credito = X_risco_credito.astype(int)

dados_codificados = pd.DataFrame(
    X_risco_credito,
    columns=['historia', 'divida', 'garantias', 'renda']
)
dados_codificados['risco'] = y_risco_credito
dados_codificados

,historia,divida,garantias,renda,risco
0,2,0,1,0,alto
1,1,0,1,1,alto
2,1,1,1,1,moderado
3,1,1,1,2,alto
4,1,1,1,2,baixo
5,1,1,0,2,baixo
6,2,1,1,0,alto
7,2,1,0,2,moderado
8,0,1,1,2,baixo
9,0,0,0,2,baixo


In [6]:
# Exibição dos códigos atribuídos a cada categoria.
encoders = {
    'historia': label_encoder_historia,
    'divida': label_encoder_divida,
    'garantias': label_encoder_garantias,
    'renda': label_encoder_renda
}

for atributo, encoder in encoders.items():
    mapeamento = dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))
    print(f'{atributo}: {mapeamento}')

historia: {'boa': np.int64(0), 'desconhecida': np.int64(1), 'ruim': np.int64(2)}
divida: {'alta': np.int64(0), 'baixa': np.int64(1)}
garantias: {'adequada': np.int64(0), 'nenhuma': np.int64(1)}
renda: {'0_15': np.int64(0), '15_35': np.int64(1), 'acima_35': np.int64(2)}


## c) Salvamento do arquivo pré-processado

In [7]:
caminho_pkl = caminho_csv.with_name('risco_credito.pkl')
with caminho_pkl.open('wb') as arquivo:
    pickle.dump([X_risco_credito, y_risco_credito], arquivo)

print('Arquivo risco_credito.pkl salvo com sucesso.')

Arquivo risco_credito.pkl salvo com sucesso.


# 2 — Algoritmo Naïve Bayes

## a) Treinamento do algoritmo

In [8]:
naiveb_risco_credito = GaussianNB()
naiveb_risco_credito.fit(X_risco_credito, y_risco_credito)
print('Modelo treinado com sucesso.')

Modelo treinado com sucesso.


## b) Previsões

Serão avaliados os dois clientes solicitados:

1. história boa, dívida alta, nenhuma garantia e renda acima de 35;
2. história ruim, dívida alta, garantia adequada e renda de 0 a 15.

In [9]:
novos_clientes = np.array([
    [
        label_encoder_historia.transform(['boa'])[0],
        label_encoder_divida.transform(['alta'])[0],
        label_encoder_garantias.transform(['nenhuma'])[0],
        label_encoder_renda.transform(['acima_35'])[0]
    ],
    [
        label_encoder_historia.transform(['ruim'])[0],
        label_encoder_divida.transform(['alta'])[0],
        label_encoder_garantias.transform(['adequada'])[0],
        label_encoder_renda.transform(['0_15'])[0]
    ]
])

previsoes = naiveb_risco_credito.predict(novos_clientes)
probabilidades = naiveb_risco_credito.predict_proba(novos_clientes)

resultado = pd.DataFrame({
    'cliente': ['i', 'ii'],
    'previsao': previsoes
})
resultado

,cliente,previsao
0,i,baixo
1,ii,moderado


In [10]:
probabilidades_df = pd.DataFrame(
    probabilidades,
    columns=[f'P({classe})' for classe in naiveb_risco_credito.classes_],
    index=['cliente i', 'cliente ii']
)
probabilidades_df

,P(alto),P(baixo),P(moderado)
cliente i,0.065874,0.934122,0.000004
cliente ii,0.000000,0.000000,1.000000


### Classes e quantidade de registros por classe

In [11]:
print('Classes utilizadas pelo algoritmo:', naiveb_risco_credito.classes_)
print('Quantidade de registros por classe:', naiveb_risco_credito.class_count_)

contagem_classes = pd.DataFrame({
    'classe': naiveb_risco_credito.classes_,
    'quantidade': naiveb_risco_credito.class_count_.astype(int)
})
contagem_classes

Classes utilizadas pelo algoritmo: ['alto' 'baixo' 'moderado']
Quantidade de registros por classe: [6. 5. 3.]


,classe,quantidade
0,alto,6
1,baixo,5
2,moderado,3


## Conclusão

O modelo classificou o cliente **i** como risco **baixo** e o cliente **ii** como risco **moderado**. As classes aprendidas foram `alto`, `baixo` e `moderado`, com 6, 5 e 3 registros, respectivamente.